In [22]:
import pandas as pd
import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
from extra import toTime, submit
from sklearn.metrics import mean_absolute_error, r2_score
from catboost import CatBoostRegressor

In [23]:
# Adding some supporting functions

# Extracting temporal features
def toTime(df):
    df = df.copy()
    # Convert and extract features from measured time
    df['time'] = pd.to_datetime(df['time'], errors = 'coerce')
    df.loc[:, 'hour'] = df['time'].dt.hour
    df.loc[:, 'day'] = df['time'].dt.day
    df.loc[:, 'month'] = df['time'].dt.month
    df.loc[:, 'dayofweek'] = df['time'].dt.dayofweek
    return df

# Create a DataFrame for submission
def submit(test_ids, ylong, ylat):
    # Create a submission DataFrame
    submission = pd.DataFrame({
        'ID': test_ids,  # Pass the test ID column
        'longitude_predicted': ylong,  # Your predicted longitudes
        'latitude_predicted': ylat    # Your predicted latitudes
    })

    # Ensure the correct column order
    submission = submission[['ID', 'longitude_predicted', 'latitude_predicted']]

    # Find the next available filename by checking existing files
    i = 1
    while os.path.exists(f'jakobs_results_{i}.csv'):
        i += 1

    # Save to CSV with the next available filename
    filename = f'jakobs_results_{i}.csv'
    submission.to_csv(filename, index=False)
    print(f"Results saved to {filename}")

In [24]:
# First we read the necessary csv files. NOTE: Different delimiter for some files.
df_train = pd.read_csv('/Users/jakobrudeovstaas/Desktop/Project ML/ais_train.csv', delimiter = '|')
df_test = pd.read_csv('/Users/jakobrudeovstaas/Desktop/Project ML/ais_test.csv')
df_vessels = pd.read_csv('/Users/jakobrudeovstaas/Desktop/Project ML/vessels.csv', delimiter = '|')

# NOTE: Train is sorted by vessel ID and time for model to notice temporal trend.
df_train = df_train.sort_values(by = ['vesselId', 'time'])

In [25]:
# Extracting temporal features
df_train = toTime(df_train)
df_test = toTime(df_test)

In [26]:
# Cleaning the data. Ensuring all data is within reasonable limits and replacing outliers/erranous data with NaN
df_train['cog'] = df_train['cog'].apply(lambda x: float(x) if float(x) < 360 else pd.NA)
# NOTE: We determine that SOG above 25 knots shall be disregarded.
df_train['sog'] = df_train['sog'].apply(lambda x: float(x) if float(x) < 25 else pd.NA)
df_train['heading'] = df_train['heading'].apply(lambda x: float(x) if float(x) < 360 else pd.NA)
# NOTE: Navstat of 0 and 8 both indicate a moving vessel. We therefore combine these two
df_train['navstat'] = df_train['navstat'].replace(8, 0)

In [27]:
# Finding last known values for each unique vessel ID
last_known_locations = df_train.groupby('vesselId').agg(
    last_latitude=('latitude', 'last'),
    last_longitude=('longitude', 'last'),
    last_heading = ('heading', 'last'),
    last_navstat = ('navstat', 'last')
).reset_index()

df_train = pd.merge(df_train, last_known_locations, on = 'vesselId', how = 'left')
df_test = pd.merge(df_test, last_known_locations, on = 'vesselId', how = 'left')

In [28]:
# Adding the avg SOG for each vessel ID. NOTE: mean is based on when the vessel is moving
isMoving = df_train[df_train['navstat'] == 0]
avg_sog_moving = isMoving.groupby('vesselId')['sog'].mean()
df_train['avg_sog_moving'] = df_train['vesselId'].map(avg_sog_moving)
df_test['avg_sog_moving'] = df_test['vesselId'].map(avg_sog_moving)

In [29]:
print(df_train.columns)

Index(['time', 'cog', 'sog', 'rot', 'heading', 'navstat', 'etaRaw', 'latitude',
       'longitude', 'vesselId', 'portId', 'hour', 'day', 'month', 'dayofweek',
       'last_latitude', 'last_longitude', 'last_heading', 'last_navstat',
       'avg_sog_moving'],
      dtype='object')


In [32]:
# Implementing a CatBoostRegressor. 
# NOTE: XGBRegressor should also be considered but CatBoost diminishes the need for label encoding

# First defining training, test sets and target predictors.
features = ['vesselId', 'hour', 'day', 'month', 'dayofweek', 'last_latitude', 'last_longitude', 'last_heading', 'avg_sog_moving']
categorical = ['vesselId', 'navstat'] # defining categorical features for model
X_train = df_train[features]
X_test = df_test[features]
y_lat = df_train['latitude']
y_long = df_train['longitude']

# Defining the model
CBlat = CatBoostRegressor(cat_features = categorical)
CBlong = CatBoostRegressor(cat_features = categorical)

# Training the model
CBlat.fit(X_train, y_lat)
CBlong.fit(X_train, y_long)

y_pred_lat = CBlat.fit(X_test)
y_pred_Long = CBlong.fit(X_test)